# agentfw -- the defense grid only (free Colab T4)

Only what is still left to compute. Everything already finished -- the English baseline, all
capability probes, the Hindi-environment probe -- is **not** in this notebook; their results
come back from Drive (`cache.tgz`) in step 4 and the grid reads them from there.

**Each session:**
1. **Runtime -> Change runtime type -> T4 GPU**
2. **Runtime -> Run all**
3. Leave it. Step 8 prints what is done and what is left, then works through what is left.
   When Colab ends the session, that is expected: progress is saved to Drive every 15 minutes.
4. On the laptop: download `MyDrive/agentfw/cache.tgz` and tell Claude.

When step 8 prints **"nothing left"**, the grid is complete -- stop using this notebook.

Setup (steps 1-7) takes ~20 minutes each session: installing vLLM and downloading the model
(~10 GB) cannot be skipped on a fresh Colab machine.


## 1 · GPU check


In [ ]:
!nvidia-smi
import subprocess, sys
try:
    cap = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader"],
        text=True).strip()
    print("\nGPU:", cap)
    if "7.5" in cap:
        print("Turing (sm75). The pinned vLLM version below is chosen for this.")
except Exception as e:
    sys.exit(f"No GPU visible ({e}). Runtime -> Change runtime type -> T4 GPU.")

## 2 · Config


In [ ]:
# Model. 14B AWQ 4-bit (~9.3 GiB of weights). The 7B baseline (RESULTS.md, 2026-10-02)
# completed only 30% of tasks and 44% of attacker goals, leaving 5 successful attacks:
# too few to compare defenses. 14B is the same Qwen2.5 family, so the same vLLM works.
MODEL     = "Qwen/Qwen2.5-14B-Instruct-AWQ"
QUANT     = "awq"
REVISION  = "main"          # pin a commit SHA here once you have one, then never move it

# Memory on one 16 GB T4 is tight for 14B: weights ~9.3 GiB leave ~3-4 GiB of KV cache.
# 12288 tokens still covers the largest prompts (workspace/travel tool schemas ~4-5k);
# --enforce-eager skips CUDA-graph capture, which would otherwise take ~1 GiB.
MAX_LEN    = 12288
GPU_UTIL   = 0.95
EXTRA_ARGS = ["--enforce-eager"]

# Fallback if 14B will not load: the 7B AWQ build, which is known to work on this setup.
# If the fallback is used, say so in RESULTS.md -- it changes the capability ceiling.
FALLBACK_MODEL = "Qwen/Qwen2.5-7B-Instruct-AWQ"
FALLBACK_QUANT = "awq"

# vLLM pinned for Turing. If this install fails, try "vllm==0.7.3", then plain "vllm".
VLLM_SPEC = "vllm==0.6.6.post1"

# Attack. AgentDojo's standard template; "direct" = bare goal, as the first baselines ran.
ATTACK = "important_instructions"

# Every step below skips work whose results already exist for this model, attack and
# corpus. Set True only to deliberately recompute (the response cache still applies).
RERUN = False

# Run shape
WORKERS = 8          # vLLM batches continuously, so concurrency is what buys throughput
SMOKE_N = 10         # cases in the pre-flight gate
FULL_N  = None       # None = all 150

# Paths
REPO_DIR      = "/content/agentfw"
DRIVE_DIR     = "/content/drive/MyDrive/agentfw"   # cache + results persist here
USE_DRIVE     = True
GIT_URL       = ""   # e.g. "https://github.com/you/agentfw.git"; blank = zip or upload
AGENTDOJO_SHA = "089ed468cf3ed0322acc66b0211f26d9d90dbf60"

# The response cache keys on the model id, so changing MODEL starts the cache cold.
print("model:", MODEL, "| workers:", WORKERS, "| cases:", FULL_N or "all")

## 3 · Drive


In [ ]:
import os
DRIVE_OK = False
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        os.makedirs(DRIVE_DIR, exist_ok=True)
        DRIVE_OK = True
        print("drive ready:", DRIVE_DIR)
        print("contents:", os.listdir(DRIVE_DIR) or "(empty)")
    except Exception as e:
        print("drive unavailable, continuing without persistence:", e)
else:
    print("drive disabled by config")

## 4 · Get the repo, restore saved progress from Drive


In [ ]:
import os, shutil, subprocess, zipfile, glob

def _flatten(dest):
    """If the zip had a single wrapper folder, lift its contents up."""
    entries = [e for e in os.listdir(dest) if e != "__MACOSX"]
    if len(entries) == 1 and os.path.isdir(os.path.join(dest, entries[0])):
        inner = os.path.join(dest, entries[0])
        if os.path.exists(os.path.join(inner, "pyproject.toml")):
            for e in os.listdir(inner):
                shutil.move(os.path.join(inner, e), os.path.join(dest, e))
            os.rmdir(inner)

if os.path.exists(os.path.join(REPO_DIR, "pyproject.toml")):
    print("repo already present at", REPO_DIR)
elif GIT_URL:
    subprocess.run(["git", "clone", "--depth", "1", GIT_URL, REPO_DIR], check=True)
else:
    zips = sorted(glob.glob(os.path.join(DRIVE_DIR, "*.zip"))) if DRIVE_OK else []
    if zips:
        src = zips[0]
        print("unzipping", src)
    else:
        from google.colab import files
        print("Upload a zip of the agentfw project folder:")
        up = files.upload()
        src = "/content/" + next(iter(up))
    os.makedirs(REPO_DIR, exist_ok=True)
    with zipfile.ZipFile(src) as z:
        z.extractall(REPO_DIR)
    _flatten(REPO_DIR)

assert os.path.exists(os.path.join(REPO_DIR, "harness", "runner.py")), \
    f"{REPO_DIR} does not look like agentfw: {os.listdir(REPO_DIR)}"

# Restore the cache and traces from a previous session, if any.
tgz = os.path.join(DRIVE_DIR, "cache.tgz") if DRIVE_OK else ""
if tgz and os.path.exists(tgz):
    subprocess.run(["tar", "xzf", tgz, "-C", REPO_DIR], check=True)
    print("cache restored from", tgz)
else:
    print("no cache.tgz in Drive — starting cold (expected on the first session)")

os.chdir(REPO_DIR)
print("cwd:", os.getcwd())
print(sorted(os.listdir()))

def save_to_drive(note=""):
    """Push cache, traces and results to Drive. Called after every long step, so a
    disconnect or an exhausted quota costs at most the step that was running."""
    import glob, re
    subprocess.run(["tar", "czf", "/content/cache.tgz", ".cache", "traces", "analysis/out"], check=False)
    size = os.path.getsize("/content/cache.tgz") / 1e6
    if not DRIVE_OK:
        print(f"cache.tgz {size:.1f} MB (Drive not mounted -- run the last cell to download it)")
        return
    # copy then rename: a session killed mid-copy must not leave a half-written archive
    subprocess.run(["cp", "/content/cache.tgz", f"{DRIVE_DIR}/cache.tgz.tmp"], check=True)
    os.replace(f"{DRIVE_DIR}/cache.tgz.tmp", f"{DRIVE_DIR}/cache.tgz")
    served = globals().get("SERVED", MODEL)
    dest = f"{DRIVE_DIR}/results/{re.sub(r'[^A-Za-z0-9._-]+', '-', served)}"
    os.makedirs(dest, exist_ok=True)
    files = glob.glob("analysis/out/phase1_*") + glob.glob("analysis/out/phase2_probe_*.json")
    for f in files:
        subprocess.run(["cp", f, f"{dest}/"], check=False)
    print(f"saved to Drive{' (' + note + ')' if note else ''}: cache.tgz {size:.1f} MB, {len(files)} result files -> {dest}")


## 5 · Install (5-10 min)


In [ ]:
import subprocess, sys, os

# Colab's own Python moves (3.13 as of Oct 2026) and vLLM pinned for the T4 needs
# <=3.12, so everything runs in a separate Python 3.10 environment. Every later
# cell launches programs with `virtual_env_python`, never `sys.executable`.
VENV = "/content/venv"
virtual_env_python = f"{VENV}/bin/python"
LOCK = f"{DRIVE_DIR}/requirements-colab.lock" if DRIVE_OK else None

def run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode:
        print("".join((r.stdout + r.stderr).splitlines(keepends=True)[-40:]))
        raise RuntimeError(f"failed: {' '.join(cmd)} -- see the output above")
    return r.stdout

def stream(cmd):
    """Run a program and show its output in this cell as it runs.

    Colab does not display output from a plain subprocess.run(), so a failing
    script would show nothing but its exit code.
    """
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         bufsize=1, env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in p.stdout:
        print(line, end="")
    return subprocess.CompletedProcess(cmd, p.wait())

def pip(*args):
    run([virtual_env_python, "-m", "pip", "install", "-q", *args])

# 1. Python 3.10. uv downloads a standalone interpreter, so this keeps working
#    even on a Colab image whose apt no longer carries python3.10.
if not os.path.exists(virtual_env_python):
    print("creating the Python 3.10 environment ...")
    run(["python3", "-m", "pip", "install", "-q", "uv"])
    run(["python3", "-m", "uv", "venv", "--seed", "--python", "3.10", VENV])
print("Colab Python:", sys.version.split()[0])
print("venv Python: ", run([virtual_env_python, "--version"]).strip())

# 2. AgentDojo at its pinned commit (vendor/ is gitignored, so a fresh copy has none).
if not os.path.exists("vendor/agentdojo/pyproject.toml"):
    os.makedirs("vendor", exist_ok=True)
    run(["git", "clone", "--quiet", "--depth", "50",
         "https://github.com/ethz-spylab/agentdojo.git", "vendor/agentdojo"])
    run(["git", "-C", "vendor/agentdojo", "checkout", "--quiet", AGENTDOJO_SHA])
sha = run(["git", "-C", "vendor/agentdojo", "rev-parse", "HEAD"]).strip()
assert sha == AGENTDOJO_SHA, f"AgentDojo at {sha}, expected {AGENTDOJO_SHA}"
print("agentdojo pinned at", sha)

# 3. Packages. The first good session writes a lock file to Drive; every later
#    session installs exactly those versions, so an upstream release can never
#    break a later phase. Delete the lock in Drive only to deliberately upgrade.
if LOCK and os.path.exists(LOCK):
    locked = open(LOCK).read()
    t = [l for l in locked.splitlines() if l.lower().startswith("transformers==")]
    if not t or not t[0].split("==")[1].startswith("4."):
        print("discarding", LOCK, "-- it records", t or "no transformers", "which breaks vLLM 0.6.6")
        os.remove(LOCK)
if LOCK and os.path.exists(LOCK):
    print("installing the exact versions from", LOCK, "...")
    pip("-r", LOCK)
else:
    # vLLM first: it pins torch, and anything installed before it could be stomped.
    print(f"installing {VLLM_SPEC} and dependencies (first time: 5-10 min) ...")
    # vLLM 0.6.x declares transformers>=4.45.2 with no upper bound, and
    # transformers 5.x removed `all_special_tokens_extended`, which this vLLM
    # still reads -- the server then dies loading the tokenizer. Pin the
    # transformers of vLLM 0.6.6's own era, in the same resolve as vLLM.
    pip(VLLM_SPEC, "transformers>=4.45.2,<4.48")
    # AgentDojo's own dependency list drags in langchain for a pipeline we do not
    # use, so it goes in with --no-deps and its real runtime imports are listed
    # here. agentdojo.agent_pipeline imports every provider SDK at load time,
    # hence anthropic/cohere/google-genai: client libraries only, no paid calls.
    VLLM_PINS = "/content/vllm-constraints.txt"
    with open(VLLM_PINS, "w") as f:
        f.write(run([virtual_env_python, "-m", "pip", "freeze", "--exclude-editable"]))
    pip("-c", VLLM_PINS, "pydantic[email]>=2.7.1", "docstring-parser>=0.15", "tenacity>=8.2.3",
        "pyyaml>=6.0.1", "click>=8.1.7", "deepdiff>=8.6.1", "rich>=13.7.1",
        "python-dotenv>=1.0.1", "typing-extensions>=4.11.0", "requests",
        "anthropic>=0.47.0", "cohere>=5.3.4", "google-genai>=1.15.0")
pip("--no-deps", "-e", "vendor/agentdojo")
pip("--no-deps", "-e", ".")

# 4. Import check before spending GPU minutes on a broken environment.
out = run([virtual_env_python, "-c",
    "from agentdojo.task_suite.load_suites import get_suites;"
    "s=get_suites('v1.2.1');"
    "from harness.runner import AgentLoop, score_case;"
    "from harness.local_client import OpenAICompatClient;"
    "print('suites:', sorted(s))"])
print(out)

# 5. Snapshot these versions as a *candidate*. An import check is not enough to
#    call them good (the transformers 5.x break passed it), so the lock is only
#    written to Drive by cell 7, after a real model has loaded and called a tool.
CANDIDATE_LOCK = "/content/requirements-candidate.lock"
with open(CANDIDATE_LOCK, "w") as f:
    f.write(run([virtual_env_python, "-m", "pip", "freeze", "--exclude-editable"]))
print("transformers", run([virtual_env_python, "-c", "import transformers; print(transformers.__version__)"]).strip())

import json as _json
def baseline_done():
    """The Phase 1 result for this model + attack, if a complete one already exists."""
    f = f"analysis/out/phase1_en_baseline__{ATTACK}.json"
    if RERUN or not os.path.exists(f):
        return None
    d = _json.load(open(f, encoding="utf-8"))
    ok = (d.get("cell", {}).get("model") == SERVED and len(d.get("attack", [])) == 150
          and sum(r.get("errored", False) for r in d["attack"]) <= 15)
    return f if ok else None


## 6 · Serve the model (first boot downloads ~10 GB)


In [ ]:
import subprocess, time, requests, os, sys

LOG = "/content/vllm.log"

def serve(model, quant):
    cmd = [virtual_env_python, "-m", "vllm.entrypoints.openai.api_server",
           "--model", model,
           "--revision", REVISION,
           "--dtype", "half",                       # T4 has no bfloat16
           "--max-model-len", str(MAX_LEN),
           "--gpu-memory-utilization", str(GPU_UTIL),
           "--enable-auto-tool-choice",
           "--tool-call-parser", "hermes",          # Qwen2.5 speaks the Hermes format
           "--disable-log-requests",
           "--port", "8000"]
    if quant:
        cmd += ["--quantization", quant]
    cmd += EXTRA_ARGS
    print(" ".join(cmd), "\n")
    return subprocess.Popen(cmd, stdout=open(LOG, "w"), stderr=subprocess.STDOUT)

def wait(proc, minutes=25):
    deadline = time.time() + minutes * 60
    while time.time() < deadline:
        if proc.poll() is not None:
            return False
        try:
            if requests.get("http://localhost:8000/v1/models", timeout=3).ok:
                return True
        except Exception:
            pass
        time.sleep(10)
    return False

def tail(n=40):
    print("".join(open(LOG).readlines()[-n:]) if os.path.exists(LOG) else "(no log)")

# A server left over from an earlier run of this cell would answer the health
# check below and hide whether the new one actually started.
subprocess.run(["pkill", "-f", "vllm.entrypoints.openai.api_server"])
time.sleep(5)

SERVED = MODEL
server = serve(MODEL, QUANT)
if wait(server):
    print("server up:", SERVED)
else:
    print(f"--- {MODEL} did not come up; log tail ---")
    tail()
    try: server.kill(); server.wait(timeout=30)
    except Exception: pass
    print(f"\nfalling back to {FALLBACK_MODEL}")
    SERVED = FALLBACK_MODEL
    server = serve(FALLBACK_MODEL, FALLBACK_QUANT)
    if not wait(server):
        tail()
        raise RuntimeError("no server. Read the log above before changing anything else.")
    print("server up on the fallback model. Record this in RESULTS.md.")

print("\nSERVED =", SERVED)

## 7 · Verify tool calling (seconds; never skip)


In [ ]:
try:
    from openai import OpenAI
except ImportError:
    import subprocess
    subprocess.run(["python3", "-m", "pip", "install", "-q", "openai"], check=True)
    from openai import OpenAI
c = OpenAI(base_url="http://localhost:8000/v1", api_key="EMPTY")

r = c.chat.completions.create(
    model=SERVED,
    messages=[{"role": "user", "content": "What is my current account balance?"}],
    tools=[{"type": "function", "function": {
        "name": "get_balance",
        "description": "Get the account balance",
        "parameters": {"type": "object", "properties": {}},
    }}],
    tool_choice="auto",
    temperature=0.0,
    max_tokens=128,
)
calls = r.choices[0].message.tool_calls
print("tool_calls:", calls)
print("text:", repr(r.choices[0].message.content))
if not calls:
    tail(60)
    raise RuntimeError(
        "No tool call. Nothing downstream is meaningful. Check --enable-auto-tool-choice "
        "and --tool-call-parser hermes in the log above before running anything else."
    )
print("\nOK — the model calls tools through the OpenAI wire format.")

# The environment has now served a real model end to end: lock its versions.
import os, shutil
if LOCK and not os.path.exists(LOCK) and os.path.exists(CANDIDATE_LOCK):
    shutil.copy(CANDIDATE_LOCK, LOCK)
    print("locked known-good package versions to", LOCK)

## 8 · The grid: what is left, in priority order

Prints done / left first, then runs only what is left. Hindi defended cells run on the
scorable cases only (~40 of 150). Autosave to Drive every 15 minutes.


In [ ]:
import os, time, subprocess, threading

# Autosave: a step can run 1.5 h, and a session cut off mid-step used to lose all of it
# (spotlight, 2026-10-07). Every 15 min the response cache goes to Drive, so a killed step
# resumes from cache next session. One lock, so the autosave and the end-of-step save
# never write /content/cache.tgz at the same time.
if not getattr(save_to_drive, "_locked", False):
    _save_lock, _save_unlocked = threading.Lock(), save_to_drive
    def save_to_drive(note=""):
        with _save_lock:
            _save_unlocked(note)
    save_to_drive._locked = True
_autosave_stop = threading.Event()
def _autosave():
    while not _autosave_stop.wait(15 * 60):
        save_to_drive("autosave")
threading.Thread(target=_autosave, daemon=True).start()
pip("sentencepiece", "protobuf")   # DeBERTa-v3 tokenizer, for the protectai steps

PLAN = [
    ("hi", "undefended"),          # all 150 cases: raw ASR + the existence proof need them
    ("en", "spotlight"),
    ("hi", "sandwich"),
    ("en", "camel_lite"),
    ("hi", "camel_lite"),
    ("en", "allowlist"),
    ("en", "protectai_deberta"),   # Class A in the loop, default threshold 0.5
    ("hi", "protectai_deberta"),
    ("en", "honeytool"),
    # ("en", "hierarchy"),         # dropped 2026-10-07 (de-scoping ladder): sandwich + spotlight
                                   # already cover Class B; re-enable only if time is left over
]

r = subprocess.run([virtual_env_python, "-c",
                    "from corpus.attacks import TEMPLATE_STATUS as T; print(bool(T['hi']['verified_by']))"],
                   capture_output=True, text=True)
HI_READY = r.stdout.strip() == "True"
if not HI_READY:
    print("Hindi steps SKIPPED: the Hindi attack template is not human-verified yet "
          "(corpus/review/templates.txt). English steps still run.\n")

def _done(lang, d):
    return os.path.exists(f"analysis/out/grid/{lang}__{d}__{ATTACK}.json")

_left = [(l, d) for l, d in PLAN if not _done(l, d)]
print("DONE :", ", ".join(f"{l} {d}" for l, d in PLAN if _done(l, d)) or "-")
print("LEFT :", ", ".join(f"{l} {d}" for l, d in _left) or "nothing left -- the grid is complete")
print()

for lang, d in PLAN:
    if lang == "hi" and not HI_READY:
        continue
    if os.path.exists(f"analysis/out/grid/{lang}__{d}__{ATTACK}.json") and not RERUN:
        continue
    argv = [virtual_env_python, "-m", "scripts.phase5_grid", "--langs", lang, "--defenses", d,
            "--model", SERVED, "--attack", ATTACK, "--workers", str(WORKERS),
            "--classifier-device", "cpu"]
    if lang != "en":
        argv.append("--scorable-only")
    t0 = time.time()
    r = stream(argv)
    print(f"[{lang} {d}] exit {r.returncode} after {(time.time() - t0) / 60:.1f} min")
    save_to_drive(f"grid {lang} {d}")

for lang in ("en", "hi"):
    ds = ["undefended"] + [d for l, d in PLAN if l == lang and d != "undefended"]
    if lang == "en":
        ds.insert(1, "sandwich")
    stream([virtual_env_python, "-m", "scripts.phase5_grid", "--summary-only",
            "--langs", lang, "--defenses", *ds, "--attack", ATTACK])

_autosave_stop.set()


## 9 · Final save


In [ ]:
save_to_drive("final")
if not DRIVE_OK:
    from google.colab import files
    files.download("/content/cache.tgz")
